# Cross-validation Generic ETC vs Astropy - Tous les instruments

Ce notebook teste la cross-validation sur tous les instruments et affiche les résultats visuellement.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.gridspec import GridSpec

# Import Generic ETC
sys.path.insert(0, 'notebooks')
from Observation import Observation, load_instruments
from astropy.stats import signal_to_noise_oir_ccd

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
def test_instrument_variations(instruments, instrument_name, verbose=False):
    """
    Test cross-validation avec variations pour un instrument
    Returns: dict avec résultats et variations
    """
    # Extract parameters
    params = {}
    for i, charact in enumerate(instruments['Charact.']):
        if charact and not isinstance(charact, np.ma.core.MaskedConstant):
            value = instruments[instrument_name][i]
            if not isinstance(value, np.ma.core.MaskedConstant):
                params[charact] = value

    try:
        # Get readout time
        readout_time = params.get('Readout', 0.0)
        
        results = {'variations': []}
        
        # Test variations exposure time
        for t_exp in [100, 500, 1000, 2000, 5000]:
            acq_time = (t_exp + readout_time) / 3600.0
            
            obs = Observation(
                instruments=instruments, instrument=instrument_name,
                exposure_time=t_exp, SNR_res="per pix", IFS=False, test=True,
                acquisition_time=acq_time
            )
            
            source_eps = obs.Signal_el[obs.i] / t_exp
            sky_eps = obs.sky[obs.i] / t_exp
            dark_eps = obs.Dark_current_f[obs.i] / t_exp
            
            snr_astropy = signal_to_noise_oir_ccd(
                t=t_exp, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
                rd=params['RN'], npix=int(obs.number_pixels_used), gain=1.0
            )
            
            results['variations'].append({
                'param': 'exposure_time',
                'value': t_exp,
                'snr_generic': obs.SNR[obs.i],
                'snr_astropy': snr_astropy,
                'ratio': obs.SNR[obs.i] / snr_astropy
            })
        
        # Test variations sky
        for factor in [0.1, 0.5, 1.0, 2.0, 5.0]:
            idx = list(instruments['Charact.']).index('Sky')
            original = instruments[instrument_name][idx]
            instruments[instrument_name][idx] = params['Sky'] * factor
            
            t_exp = 1000.0
            acq_time = (t_exp + readout_time) / 3600.0
            
            obs = Observation(
                instruments=instruments, instrument=instrument_name,
                exposure_time=t_exp, SNR_res="per pix", IFS=False, test=True,
                acquisition_time=acq_time
            )
            
            source_eps = obs.Signal_el[obs.i] / t_exp
            sky_eps = obs.sky[obs.i] / t_exp
            dark_eps = obs.Dark_current_f[obs.i] / t_exp
            
            snr_astropy = signal_to_noise_oir_ccd(
                t=t_exp, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
                rd=params['RN'], npix=int(obs.number_pixels_used), gain=1.0
            )
            
            instruments[instrument_name][idx] = original
            
            results['variations'].append({
                'param': 'sky',
                'value': factor,
                'snr_generic': obs.SNR[obs.i],
                'snr_astropy': snr_astropy,
                'ratio': obs.SNR[obs.i] / snr_astropy
            })
        
        df = pd.DataFrame(results['variations'])
        results['instrument'] = instrument_name
        results['success'] = True
        results['error'] = None
        results['ratio_mean'] = df['ratio'].mean()
        results['ratio_std'] = df['ratio'].std()
        results['df'] = df
        
        return results
        
    except Exception as e:
        if verbose:
            print(f"{instrument_name}: ERROR - {str(e)}")
        return {
            'instrument': instrument_name,
            'success': False,
            'error': str(e),
            'variations': []
        }

In [ ]:
# Load instruments
print("Chargement des instruments...")
instruments, database = load_instruments()

# Get instrument list
metadata_cols = {'.', 'Charact.', 'Unit'}
instrument_names = [key for key in instruments.keys() if key not in metadata_cols]

print(f"\n{len(instrument_names)} instruments trouvés")
print("\nTest en cours...\n")

In [ ]:
# Test tous les instruments
all_results = []

for i, inst_name in enumerate(instrument_names):
    print(f"[{i+1}/{len(instrument_names)}] {inst_name}...", end=" ")
    result = test_instrument_variations(instruments, inst_name, verbose=False)
    all_results.append(result)
    
    if result['success']:
        print(f"✓ ratio={result['ratio_mean']:.4f}")
    else:
        print(f"✗ {result['error'][:50]}")

print("\nTests terminés!")

## Résumé général

In [ ]:
# Summary statistics
successful = [r for r in all_results if r['success']]
failed = [r for r in all_results if not r['success']]

print("="*80)
print("RÉSUMÉ GÉNÉRAL")
print("="*80)
print(f"\nInstruments testés: {len(instrument_names)}")
print(f"  - Succès: {len(successful)}")
print(f"  - Échecs: {len(failed)}")

if len(successful) > 0:
    ratios = [r['ratio_mean'] for r in successful if not np.isnan(r['ratio_mean'])]
    if len(ratios) > 0:
        print(f"\nRatio moyen: {np.mean(ratios):.6f}")
        print(f"Écart-type: {np.std(ratios):.6f}")
        print(f"Min: {np.min(ratios):.6f}")
        print(f"Max: {np.max(ratios):.6f}")
        
        # Instruments with good ratio
        good = [r for r in successful if 0.95 <= r['ratio_mean'] <= 1.05 and not np.isnan(r['ratio_mean'])]
        print(f"\n✓ Instruments avec ratio entre 0.95 et 1.05: {len(good)}/{len(successful)}")
        
        # Instruments with bad ratio
        bad = [r for r in successful if (r['ratio_mean'] < 0.95 or r['ratio_mean'] > 1.05) and not np.isnan(r['ratio_mean'])]
        if len(bad) > 0:
            print(f"\n⚠ Instruments avec ratio suspect:")
            for r in bad:
                print(f"  - {r['instrument']}: {r['ratio_mean']:.4f}")

if len(failed) > 0:
    print(f"\n❌ Instruments en échec:")
    for r in failed:
        print(f"  - {r['instrument']}: {r['error'][:60]}")

## Vue d'ensemble: distribution des ratios

In [ ]:
# Plot ratio distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ratios = [r['ratio_mean'] for r in successful if not np.isnan(r['ratio_mean'])]
axes[0].hist(ratios, bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Ratio idéal = 1.0')
axes[0].axvline(0.95, color='orange', linestyle=':', linewidth=1, label='Limites ±5%')
axes[0].axvline(1.05, color='orange', linestyle=':', linewidth=1)
axes[0].set_xlabel('Ratio SNR_Generic/SNR_Astropy', fontsize=12)
axes[0].set_ylabel('Nombre d\'instruments', fontsize=12)
axes[0].set_title('Distribution des ratios', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bar plot per instrument
inst_names = [r['instrument'] for r in successful if not np.isnan(r['ratio_mean'])]
ratios_sorted = sorted(zip(inst_names, ratios), key=lambda x: x[1])
names_sorted = [x[0] for x in ratios_sorted]
ratios_sorted_values = [x[1] for x in ratios_sorted]

colors = ['green' if 0.95 <= r <= 1.05 else 'orange' for r in ratios_sorted_values]
axes[1].barh(range(len(names_sorted)), ratios_sorted_values, color=colors, alpha=0.7)
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Ratio idéal = 1.0')
axes[1].axvline(0.95, color='orange', linestyle=':', linewidth=1)
axes[1].axvline(1.05, color='orange', linestyle=':', linewidth=1)
axes[1].set_yticks(range(len(names_sorted)))
axes[1].set_yticklabels(names_sorted, fontsize=8)
axes[1].set_xlabel('Ratio SNR_Generic/SNR_Astropy', fontsize=12)
axes[1].set_title('Ratio par instrument', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')
axes[1].legend()

plt.tight_layout()
plt.show()

## Plots détaillés par instrument

Affiche les variations pour chaque instrument qui a réussi les tests.

In [ ]:
def plot_instrument_variations(result):
    """
    Plot variations pour un instrument
    """
    if not result['success'] or len(result['variations']) == 0:
        return
    
    df = result['df']
    
    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(1, 3, figure=fig)
    
    # Exposure time variations
    ax1 = fig.add_subplot(gs[0, 0])
    df_exp = df[df['param'] == 'exposure_time']
    if len(df_exp) > 0:
        ax1.plot(df_exp['value'], df_exp['snr_generic'], 'o-', label='Generic ETC', linewidth=2, markersize=8)
        ax1.plot(df_exp['value'], df_exp['snr_astropy'], 's-', label='Astropy', linewidth=2, markersize=8)
        ax1.set_xlabel('Exposure time (s)', fontsize=11)
        ax1.set_ylabel('SNR', fontsize=11)
        ax1.set_title('Variation temps d\'exposition', fontsize=12, fontweight='bold')
        ax1.legend()
        ax1.grid(alpha=0.3)
        ax1.set_xscale('log')
        ax1.set_yscale('log')
    
    # Sky variations
    ax2 = fig.add_subplot(gs[0, 1])
    df_sky = df[df['param'] == 'sky']
    if len(df_sky) > 0:
        ax2.plot(df_sky['value'], df_sky['snr_generic'], 'o-', label='Generic ETC', linewidth=2, markersize=8)
        ax2.plot(df_sky['value'], df_sky['snr_astropy'], 's-', label='Astropy', linewidth=2, markersize=8)
        ax2.set_xlabel('Sky factor', fontsize=11)
        ax2.set_ylabel('SNR', fontsize=11)
        ax2.set_title('Variation sky background', fontsize=12, fontweight='bold')
        ax2.legend()
        ax2.grid(alpha=0.3)
        ax2.set_xscale('log')
        ax2.set_yscale('log')
    
    # Ratio variations
    ax3 = fig.add_subplot(gs[0, 2])
    if len(df_exp) > 0:
        ax3.plot(df_exp['value'], df_exp['ratio'], 'o-', label='Exp time', linewidth=2, markersize=8)
    if len(df_sky) > 0:
        ax3.plot(df_sky['value'], df_sky['ratio'], 's-', label='Sky', linewidth=2, markersize=8)
    ax3.axhline(1.0, color='red', linestyle='--', linewidth=2, label='Ratio idéal = 1.0')
    ax3.axhline(0.95, color='orange', linestyle=':', linewidth=1)
    ax3.axhline(1.05, color='orange', linestyle=':', linewidth=1)
    ax3.set_xlabel('Variation factor', fontsize=11)
    ax3.set_ylabel('Ratio (Generic/Astropy)', fontsize=11)
    ax3.set_title('Ratio SNR', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(alpha=0.3)
    ax3.set_xscale('log')
    ax3.set_ylim([0.9, 1.1])
    
    # Overall title
    fig.suptitle(f"{result['instrument']} - Ratio moyen: {result['ratio_mean']:.4f} ± {result['ratio_std']:.4f}",
                 fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()

# Plot pour chaque instrument réussi
for result in successful:
    if not np.isnan(result['ratio_mean']):
        plot_instrument_variations(result)

## Table récapitulative

In [ ]:
# Create summary table
summary_data = []
for r in all_results:
    if r['success']:
        summary_data.append({
            'Instrument': r['instrument'],
            'Ratio moyen': r['ratio_mean'],
            'Écart-type': r['ratio_std'],
            'Status': '✓ OK' if 0.95 <= r['ratio_mean'] <= 1.05 and not np.isnan(r['ratio_mean']) else '⚠ Suspect'
        })
    else:
        summary_data.append({
            'Instrument': r['instrument'],
            'Ratio moyen': np.nan,
            'Écart-type': np.nan,
            'Status': '✗ Échec'
        })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Ratio moyen', ascending=False, na_position='last')

print("\n" + "="*80)
print("TABLE RÉCAPITULATIVE")
print("="*80)
print(summary_df.to_string(index=False))